# Architecture 05: Multi-Model Router

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajvermatx/careeralign.com/blob/main/genai-arch/notebooks/05-multi-model-router.ipynb)

**Companion notebook for**: `05-multi-model-router.html`

## Learning Objectives
- Classify query complexity to route to appropriate models
- Build a router that balances cost, latency, and quality
- Implement fallback chains for reliability
- Track per-request costs and latency
- Compare model outputs with A/B testing

## Prerequisites
- Python 3.10+
- OpenAI API key (optional — mock fallbacks included)

---
## Setup & Dependencies

In [ ]:
%pip install -q openai tiktoken

In [ ]:
import os
import json
import time
import random
from dataclasses import dataclass, field

from openai import OpenAI

API_KEY = os.environ.get("OPENAI_API_KEY", "")
client = OpenAI(api_key=API_KEY) if API_KEY else None

if not API_KEY:
    print("⚠️  OPENAI_API_KEY not set. Mock responses will be used.")
else:
    print("✅ API key loaded.")

---
## Section 1: Model Configuration & Cost Table

Define the available models with their capabilities, cost per token, and speed characteristics.

In [ ]:
@dataclass
class ModelConfig:
    name: str
    model_id: str
    tier: str           # "fast", "balanced", "powerful"
    input_cost: float   # $ per 1M input tokens
    output_cost: float  # $ per 1M output tokens
    max_tokens: int
    avg_latency_ms: int # typical response time
    strengths: list[str] = field(default_factory=list)


MODELS = {
    "fast": ModelConfig(
        name="GPT-4o Mini",
        model_id="gpt-4o-mini",
        tier="fast",
        input_cost=0.15,
        output_cost=0.60,
        max_tokens=128000,
        avg_latency_ms=400,
        strengths=["simple Q&A", "classification", "formatting", "extraction"],
    ),
    "balanced": ModelConfig(
        name="GPT-4o",
        model_id="gpt-4o",
        tier="balanced",
        input_cost=2.50,
        output_cost=10.00,
        max_tokens=128000,
        avg_latency_ms=800,
        strengths=["reasoning", "analysis", "code generation", "multi-step"],
    ),
    "powerful": ModelConfig(
        name="O1",
        model_id="o1",
        tier="powerful",
        input_cost=15.00,
        output_cost=60.00,
        max_tokens=200000,
        avg_latency_ms=5000,
        strengths=["complex reasoning", "math", "research", "novel problems"],
    ),
}

# Display model comparison
print(f"{'Model':<16} {'Tier':<10} {'Input $/1M':>10} {'Output $/1M':>12} {'Latency':>10}")
print("-" * 62)
for m in MODELS.values():
    print(f"{m.name:<16} {m.tier:<10} ${m.input_cost:>8.2f} ${m.output_cost:>10.2f} {m.avg_latency_ms:>8}ms")

# Cost ratio
fast_cost = MODELS['fast'].output_cost
powerful_cost = MODELS['powerful'].output_cost
print(f"\nCost ratio (powerful/fast): {powerful_cost/fast_cost:.0f}x")
print("→ Routing simple queries to 'fast' saves significant cost.")

---
## Section 2: Complexity Classifier

Classify incoming queries by complexity to determine which model tier to use.
We use a combination of heuristics and (optionally) an LLM-based classifier.

In [ ]:
import re


def classify_complexity_heuristic(query: str) -> str:
    """Rule-based complexity classifier (fast, no API call needed)."""
    query_lower = query.lower()
    word_count = len(query.split())

    # Powerful tier indicators
    powerful_signals = [
        "prove", "derive", "theorem", "formal proof",
        "research paper", "novel approach", "compare and contrast",
        "step by step reasoning", "mathematical", "algorithm design",
        "trade-offs", "architecture", "system design",
    ]
    if any(signal in query_lower for signal in powerful_signals):
        return "powerful"

    # Balanced tier indicators
    balanced_signals = [
        "explain", "analyze", "write code", "debug",
        "implement", "refactor", "review", "optimize",
        "summarize this article", "create a plan",
    ]
    if any(signal in query_lower for signal in balanced_signals) or word_count > 50:
        return "balanced"

    # Default: fast tier
    return "fast"


# Test the classifier
test_queries = [
    "What is the capital of France?",
    "Explain the difference between TCP and UDP with examples.",
    "Write code to implement a binary search tree in Python.",
    "Prove that the halting problem is undecidable using a formal diagonalization argument.",
    "Translate 'hello' to Spanish.",
    "Design a system architecture for a real-time fraud detection platform handling 1M transactions per second.",
    "Fix this JSON: {name: 'test'}",
]

print(f"{'Query (truncated)':<60} {'Tier':>10}")
print("-" * 72)
for q in test_queries:
    tier = classify_complexity_heuristic(q)
    print(f"{q[:58]:<60} {tier:>10}")

In [ ]:
def classify_complexity_llm(query: str) -> dict:
    """LLM-based complexity classifier (more accurate, costs a small API call)."""
    prompt = f"""Classify this query's complexity for LLM routing. Return JSON:
{{"tier": "fast|balanced|powerful", "reason": "brief explanation", "confidence": 0.0-1.0}}

Guidelines:
- fast: simple lookups, formatting, translation, yes/no questions
- balanced: analysis, coding, summarization, multi-step tasks
- powerful: complex reasoning, proofs, research, novel problem-solving, system design

Query: {query}"""

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",  # Use cheapest model for routing
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=100,
            response_format={"type": "json_object"},
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        # Fallback to heuristic
        tier = classify_complexity_heuristic(query)
        return {"tier": tier, "reason": "heuristic fallback", "confidence": 0.7}


# Test LLM classifier
for q in test_queries[:3]:
    result = classify_complexity_llm(q)
    print(f"Query: {q[:50]}...")
    print(f"  → {result}\n")

---
## Section 3: Router Implementation

The core router class that classifies queries, selects the model,
calls the API, and tracks metrics.

In [ ]:
@dataclass
class RouterResult:
    query: str
    tier: str
    model: str
    response: str
    input_tokens: int
    output_tokens: int
    cost_usd: float
    latency_ms: float
    fallback_used: bool = False


class ModelRouter:
    """Routes queries to the optimal model based on complexity."""

    def __init__(self, client: OpenAI | None, models: dict[str, ModelConfig]):
        self.client = client
        self.models = models
        self.history: list[RouterResult] = []

    def _calculate_cost(self, config: ModelConfig, input_tokens: int, output_tokens: int) -> float:
        return (input_tokens * config.input_cost + output_tokens * config.output_cost) / 1_000_000

    def _call_model(self, model_id: str, messages: list[dict], max_tokens: int = 500) -> dict:
        """Call the model, return response dict or raise on failure."""
        try:
            if not self.client:
                raise Exception("No API key")
            response = self.client.chat.completions.create(
                model=model_id,
                messages=messages,
                max_tokens=max_tokens,
                temperature=0.3,
            )
            return {
                "content": response.choices[0].message.content,
                "input_tokens": response.usage.prompt_tokens,
                "output_tokens": response.usage.completion_tokens,
            }
        except Exception:
            # Mock response
            mock_tokens_in = len(messages[0]["content"].split()) * 2
            mock_tokens_out = random.randint(50, 200)
            return {
                "content": f"[Mock {model_id} response for: {messages[-1]['content'][:60]}...]",
                "input_tokens": mock_tokens_in,
                "output_tokens": mock_tokens_out,
            }

    def route(self, query: str, system_prompt: str = "You are a helpful assistant.") -> RouterResult:
        """Classify, route, and call the appropriate model."""
        # Step 1: Classify complexity
        tier = classify_complexity_heuristic(query)
        config = self.models[tier]

        # Step 2: Call model
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query},
        ]

        start = time.time()
        result = self._call_model(config.model_id, messages)
        latency = (time.time() - start) * 1000

        # Step 3: Calculate cost
        cost = self._calculate_cost(config, result["input_tokens"], result["output_tokens"])

        router_result = RouterResult(
            query=query,
            tier=tier,
            model=config.name,
            response=result["content"],
            input_tokens=result["input_tokens"],
            output_tokens=result["output_tokens"],
            cost_usd=cost,
            latency_ms=latency,
        )
        self.history.append(router_result)
        return router_result


# Create router
router = ModelRouter(client, MODELS)

# Route a few queries
for q in test_queries:
    r = router.route(q)
    print(f"[{r.tier:<9}] {r.model:<14} | ${r.cost_usd:.6f} | {r.latency_ms:6.0f}ms | {q[:50]}")

---
## Section 4: Fallback Chains

If the primary model fails (rate limit, timeout, error), automatically
fall back to the next tier.

In [ ]:
FALLBACK_CHAIN = {
    "powerful": ["powerful", "balanced", "fast"],
    "balanced": ["balanced", "fast"],
    "fast": ["fast"],
}


class ResilientRouter(ModelRouter):
    """Router with automatic fallback on failure."""

    def route_with_fallback(self, query: str, system_prompt: str = "You are a helpful assistant.") -> RouterResult:
        tier = classify_complexity_heuristic(query)
        chain = FALLBACK_CHAIN[tier]

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query},
        ]

        for i, attempt_tier in enumerate(chain):
            config = self.models[attempt_tier]
            try:
                start = time.time()
                result = self._call_model(config.model_id, messages)
                latency = (time.time() - start) * 1000

                cost = self._calculate_cost(config, result["input_tokens"], result["output_tokens"])
                router_result = RouterResult(
                    query=query,
                    tier=attempt_tier,
                    model=config.name,
                    response=result["content"],
                    input_tokens=result["input_tokens"],
                    output_tokens=result["output_tokens"],
                    cost_usd=cost,
                    latency_ms=latency,
                    fallback_used=(i > 0),
                )
                self.history.append(router_result)

                if i > 0:
                    print(f"  ⚠️  Fell back from '{tier}' to '{attempt_tier}'")

                return router_result
            except Exception as e:
                print(f"  ❌ {config.name} failed: {e}")
                continue

        # All models failed
        return RouterResult(
            query=query, tier="none", model="none",
            response="All models failed.",
            input_tokens=0, output_tokens=0, cost_usd=0, latency_ms=0,
            fallback_used=True,
        )


# Demo fallback
resilient = ResilientRouter(client, MODELS)
r = resilient.route_with_fallback("Prove the Riemann hypothesis is connected to prime distribution.")
print(f"\nRouted to: {r.model} (tier: {r.tier}, fallback: {r.fallback_used})")
print(f"Cost: ${r.cost_usd:.6f}")

---
## Section 5: Cost & Latency Tracking

Aggregate routing metrics to understand spend distribution and performance.

In [ ]:
def print_routing_report(history: list[RouterResult]):
    """Print a summary report of all routed queries."""
    if not history:
        print("No queries routed yet.")
        return

    total_cost = sum(r.cost_usd for r in history)
    total_tokens = sum(r.input_tokens + r.output_tokens for r in history)
    avg_latency = sum(r.latency_ms for r in history) / len(history)

    # Per-tier breakdown
    tier_stats = {}
    for r in history:
        if r.tier not in tier_stats:
            tier_stats[r.tier] = {"count": 0, "cost": 0.0, "latency": []}
        tier_stats[r.tier]["count"] += 1
        tier_stats[r.tier]["cost"] += r.cost_usd
        tier_stats[r.tier]["latency"].append(r.latency_ms)

    print("=" * 60)
    print("ROUTING REPORT")
    print("=" * 60)
    print(f"Total queries:  {len(history)}")
    print(f"Total cost:     ${total_cost:.6f}")
    print(f"Total tokens:   {total_tokens:,}")
    print(f"Avg latency:    {avg_latency:.0f}ms")
    print(f"Fallbacks:      {sum(1 for r in history if r.fallback_used)}")

    print(f"\n{'Tier':<12} {'Queries':>8} {'Cost':>12} {'% of Spend':>12} {'Avg Latency':>12}")
    print("-" * 58)
    for tier, stats in sorted(tier_stats.items()):
        pct = (stats['cost'] / total_cost * 100) if total_cost > 0 else 0
        avg_lat = sum(stats['latency']) / len(stats['latency'])
        print(f"{tier:<12} {stats['count']:>8} ${stats['cost']:>10.6f} {pct:>10.1f}% {avg_lat:>10.0f}ms")

    # Cost savings estimate
    all_powerful_cost = sum(
        (r.input_tokens * MODELS['powerful'].input_cost + r.output_tokens * MODELS['powerful'].output_cost) / 1_000_000
        for r in history
    )
    savings = all_powerful_cost - total_cost
    savings_pct = (savings / all_powerful_cost * 100) if all_powerful_cost > 0 else 0
    print(f"\nCost if all queries used 'powerful': ${all_powerful_cost:.6f}")
    print(f"Savings from routing:               ${savings:.6f} ({savings_pct:.0f}%)")


# Print report for the first router's history
print_routing_report(router.history)

---
## Section 6: A/B Model Comparison

Run the same query through multiple models and compare quality, cost, and speed.

In [ ]:
def ab_compare(query: str, router: ModelRouter, tiers: list[str] = None) -> list[dict]:
    """Run a query through multiple models and compare results."""
    if tiers is None:
        tiers = ["fast", "balanced"]

    results = []
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Be concise."},
        {"role": "user", "content": query},
    ]

    for tier in tiers:
        config = router.models[tier]
        start = time.time()
        result = router._call_model(config.model_id, messages, max_tokens=300)
        latency = (time.time() - start) * 1000
        cost = router._calculate_cost(config, result["input_tokens"], result["output_tokens"])

        results.append({
            "tier": tier,
            "model": config.name,
            "response": result["content"],
            "tokens": result["input_tokens"] + result["output_tokens"],
            "cost": cost,
            "latency_ms": latency,
        })

    return results


# Compare fast vs balanced on a coding question
query = "Write a Python function to check if a string is a valid palindrome, ignoring spaces and punctuation."

comparison = ab_compare(query, router, ["fast", "balanced"])

print(f"Query: {query}\n")
for r in comparison:
    print(f"--- {r['model']} ({r['tier']}) ---")
    print(f"Cost: ${r['cost']:.6f} | Latency: {r['latency_ms']:.0f}ms | Tokens: {r['tokens']}")
    print(f"Response preview: {r['response'][:150]}...")
    print()

# Cost comparison
if len(comparison) == 2:
    ratio = comparison[1]['cost'] / comparison[0]['cost'] if comparison[0]['cost'] > 0 else 0
    print(f"'{comparison[1]['model']}' costs {ratio:.1f}x more than '{comparison[0]['model']}'")
    print(f"Is the quality difference worth {ratio:.1f}x the cost? That's the routing decision.")

---
## Section 7: Budget-Aware Routing

Enforce per-request and daily budget limits. Automatically downgrade
to cheaper models when approaching budget thresholds.

In [ ]:
class BudgetRouter(ModelRouter):
    """Router that respects budget constraints."""

    def __init__(self, client, models, daily_budget: float = 10.0, per_request_max: float = 0.05):
        super().__init__(client, models)
        self.daily_budget = daily_budget
        self.per_request_max = per_request_max
        self.daily_spend = 0.0

    def _budget_adjusted_tier(self, requested_tier: str) -> str:
        """Downgrade tier if budget is running low."""
        budget_remaining = self.daily_budget - self.daily_spend
        budget_pct = budget_remaining / self.daily_budget

        if budget_pct < 0.1:  # Less than 10% budget remaining
            return "fast"  # Force cheapest model
        elif budget_pct < 0.3 and requested_tier == "powerful":
            return "balanced"  # Downgrade powerful to balanced

        return requested_tier

    def route(self, query: str, system_prompt: str = "You are a helpful assistant.") -> RouterResult:
        # Classify
        tier = classify_complexity_heuristic(query)

        # Budget adjustment
        adjusted_tier = self._budget_adjusted_tier(tier)
        if adjusted_tier != tier:
            print(f"  💰 Budget guard: downgraded '{tier}' → '{adjusted_tier}'")
            tier = adjusted_tier

        config = self.models[tier]
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query},
        ]

        start = time.time()
        result = self._call_model(config.model_id, messages)
        latency = (time.time() - start) * 1000
        cost = self._calculate_cost(config, result["input_tokens"], result["output_tokens"])

        self.daily_spend += cost

        router_result = RouterResult(
            query=query, tier=tier, model=config.name,
            response=result["content"],
            input_tokens=result["input_tokens"],
            output_tokens=result["output_tokens"],
            cost_usd=cost, latency_ms=latency,
        )
        self.history.append(router_result)
        return router_result


# Simulate budget pressure
budget_router = BudgetRouter(client, MODELS, daily_budget=0.001)  # Very tight budget for demo
budget_router.daily_spend = 0.0008  # Simulate 80% already spent

print(f"Daily budget: ${budget_router.daily_budget:.4f}")
print(f"Already spent: ${budget_router.daily_spend:.4f}")
print(f"Remaining: ${budget_router.daily_budget - budget_router.daily_spend:.4f} ({(budget_router.daily_budget - budget_router.daily_spend)/budget_router.daily_budget*100:.0f}%)\n")

# This would normally route to 'powerful' but budget forces downgrade
r = budget_router.route("Design a system architecture for a distributed cache.")
print(f"Routed to: {r.model} (tier: {r.tier})")
print(f"Spend after query: ${budget_router.daily_spend:.6f}")

---
## Summary

In this notebook we built a complete multi-model routing system:

1. **Model Configuration** — Define models with cost, latency, and capability profiles

2. **Complexity Classification** — Heuristic and LLM-based classifiers to determine query difficulty

3. **Router Core** — Routes queries to the optimal model tier and tracks metrics

4. **Fallback Chains** — Automatic failover: powerful → balanced → fast

5. **Cost Tracking** — Per-query and aggregate spend analysis with savings estimation

6. **A/B Comparison** — Run the same query through multiple models to evaluate quality vs cost

7. **Budget-Aware Routing** — Automatically downgrade tiers when approaching budget limits

**Key takeaway**: Not every query needs the most powerful model. Intelligent routing can cut costs by 60-80% with minimal quality impact for simple queries.

**Next notebook**: Architecture 06 — Agentic Tool Use